In [ ]:
from yeager_utils import surface_rv, surface_rv_ssapy, orbit_plot, RGEO, VGEO, leapfrog
import ssapy

In [ ]:
t0 = Time("2025-1-1")

In [ ]:
surface_rv(lat=0, lon=-90, t=t0), surface_rv_ssapy(lat=0, lon=-90, t=t0)

In [ ]:
orbit = Orbit.fromKeplerianElements(a=RGEO, e=.4, i=0, pa=0, raan=0, trueAnomaly=np.radians(0), t=t0)

rs, v = rv(orbit, np.arange(0, orbit.period, 600))
orbit_plot(rs, show=True)


print("Complete.")

In [ ]:
t0 = Time("2025-1-1T12:00:00.000", scale='utc')
times = get_times(duration=(24, 'hour'), freq=(1, 's'), t0=t0).gps

kwargs = dict(
    mass=250,  # [kg] --> was 1e4
    area=0.25,  # [m^2]
    CD=2.3,  # Drag coefficient
    CR=1.3,  # Radiation pressure coefficient
)

earth = ssapy.get_body("earth", model='egm2008')
moon = ssapy.get_body("moon")
sun = ssapy.get_body("sun")

# Accelerations - pass a body object or string of body name.
aEarth = ssapy.AccelKepler() + ssapy.AccelHarmonic(earth, 180, 180)
aMoon = ssapy.AccelThirdBody(moon) + ssapy.AccelHarmonic(moon)
aSun = ssapy.AccelThirdBody(sun)
aSolRad = ssapy.AccelSolRad()
aEarthRad = ssapy.AccelEarthRad()
aDrag = ssapy.AccelDrag()
accel = aEarth + aMoon + aSun + aSolRad + aEarthRad + aDrag

a = RGEO
e = 0
i = 0
pa = 0
raan = 0
trueAnomaly = 0

kElements = [a, e, i, pa, raan, trueAnomaly]

orbit = ssapy.Orbit.fromKeplerianElements(*kElements, t0, propkw=kwargs)
orbital_period = orbit.period / (60 * 60 * 24)  # in days

# Define the time indices for the maneuver in the times array
t1_index = 0  # Replace with the actual index for the start time
t2_index = 600  # Replace with the actual index for the end time
print(f"Maneuver window: {Time(times[t1_index], format='gps').ymdhms} to {Time(times[t2_index], format='gps').ymdhms}")

# Example acceleration in NTW coordinates for the maneuver
thrust = -1  # m/s
accel_T = accel + ssapy.AccelConstNTW(accelntw=[0.0, thrust, 0.0], time_breakpoints=[times[t1_index], times[t2_index]])
accel_N = accel + ssapy.AccelConstNTW(accelntw=[thrust, 0, 0.0], time_breakpoints=[times[t1_index], times[t2_index]])
accel_W = accel + ssapy.AccelConstNTW(accelntw=[0.0, 0, thrust], time_breakpoints=[times[t1_index], times[t2_index]])

rs = []
propagators = [accel, accel_N, accel_T, accel_W]

try:
    for acc in propagators:
        print(f"Running SSAPy {acc}.")
        r, _ = ssapy.rv(orbit, times, propagator=SciPyPropagator(acc))
        rs.append(r)

    print(f"Running leapfrog velocity.")
    r, _ = leapfrog(orbit.r, orbit.v, times, velocity=(t1_index, t2_index, thrust))
    rs.append(r)
    print(f"Running leapfrog radial.")
    r, _ = leapfrog(orbit.r, orbit.v, times, radial=(t1_index, t2_index, thrust))
    rs.append(r)
    print(f"Running leapfrog inclination.")
    r, _ = leapfrog(orbit.r, orbit.v, times, inclination=(t1_index, t2_index, thrust))
    rs.append(r)
except RuntimeError as err:
    print(err)


In [ ]:
import matplotlib.pyplot as plt

# Labels for each trajectory
labels = [
    "Baseline SSAPy (no thrust)",
    "Thrust: -N direction (SSAPy)",
    "Thrust: -T direction (SSAPy)",
    "Thrust: -T direction (leapfrog)",
    "Thrust: -N direction (leapfrog)"
]

# Transparency and uniform line width
alphas = [1.0] * len(labels)
linewidths = [3.0] * len(labels)

# Line styles: solid for SSAPy, dashed for leapfrog
line_styles = ['solid' if 'SSAPy' in label else 'dashed' for label in labels]

# Plot
fig, ax = plt.subplots(figsize=(8, 8))
print(f"shape of rs: {np.shape(rs)}")
for r, label, ls, alpha, lw in zip(rs, labels, line_styles, alphas, linewidths):
    ax.plot(r[:, 0], r[:, 1], label=label, linestyle=ls, alpha=alpha, linewidth=lw)

ax.set_aspect('equal')
ax.set_xlabel("X [m]")
ax.set_ylabel("Y [m]")
ax.set_title("Orbital Trajectories with Thrust Variants")
ax.legend(loc='upper right')
plt.grid(True)
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Time vector for X-axis
time_axis = times - times[0]  # seconds since t0

# Pairs to compare: (SSApy index, leapfrog index)
like_to_like_pairs = [
    (1, 4),  # -N direction
    (2, 3)   # -T direction
]

fig, ax = plt.subplots(figsize=(10, 6))

for i, j in like_to_like_pairs:
    diff = np.linalg.norm(rs[i] - rs[j], axis=1) / 1000.0  # convert to km
    label = f"{labels[i]} vs {labels[j]}"
    ax.plot(time_axis, diff, label=label, alpha=0.9)

ax.set_yscale('log')
ax.set_ylim(bottom=1)
ax.set_title("Like-to-Like Trajectory Distance Over Time")
ax.set_xlabel("Time [s since start]")
ax.set_ylabel("Distance [km]")
ax.legend(fontsize='small', loc='upper right')
ax.grid(True, which='both', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()
